<a href="https://colab.research.google.com/github/Jeevan0221/Tj-ai-portfolio/blob/main/day13_ai_database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

!pip install anthropic

In [5]:
import anthropic
import sqlite3
from google.colab import userdata
print("All files loaded")

All files loaded


In [9]:
api_key = userdata.get('ANTHROPIC_KEY')
client = anthropic.Anthropic(api_key=api_key)
print("Connected to claude")

Connected to claude


In [10]:
# Create a database file
conn = sqlite3.connect('company.db')
cursor = conn.cursor()

# Create an employees table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        department TEXT,
        salary INTEGER,
        years_experience INTEGER,
        city TEXT
    )
''')

# Add 10 employees to the database
employees_data = [
    (1, 'John Smith', 'Engineering', 145000, 8, 'New York'),
    (2, 'Sarah Johnson', 'Marketing', 95000, 5, 'Los Angeles'),
    (3, 'Mike Davis', 'Engineering', 165000, 12, 'San Francisco'),
    (4, 'Emily Brown', 'HR', 75000, 3, 'Chicago'),
    (5, 'James Wilson', 'Engineering', 125000, 6, 'New York'),
    (6, 'Lisa Anderson', 'Marketing', 110000, 7, 'Los Angeles'),
    (7, 'Robert Taylor', 'Finance', 135000, 9, 'Chicago'),
    (8, 'Jennifer Martinez', 'HR', 80000, 4, 'San Francisco'),
    (9, 'David Thomas', 'Finance', 155000, 11, 'New York'),
    (10, 'Jessica Garcia', 'Engineering', 140000, 7, 'Chicago'),
]

cursor.executemany('INSERT OR IGNORE INTO employees VALUES (?,?,?,?,?,?)', employees_data)
conn.commit()

print("Database created!")
print("10 employees added!")

Database created!
10 employees added!


In [11]:
# Read all employees from the database
cursor.execute('SELECT * FROM employees')
all_employees = cursor.fetchall()

print("=== All Employees in Database ===")
for employee in all_employees:
    print(f"ID: {employee[0]} | Name: {employee[1]} | Dept: {employee[2]} | Salary: ${employee[3]:,} | Experience: {employee[4]} years | City: {employee[5]}")


=== All Employees in Database ===
ID: 1 | Name: John Smith | Dept: Engineering | Salary: $145,000 | Experience: 8 years | City: New York
ID: 2 | Name: Sarah Johnson | Dept: Marketing | Salary: $95,000 | Experience: 5 years | City: Los Angeles
ID: 3 | Name: Mike Davis | Dept: Engineering | Salary: $165,000 | Experience: 12 years | City: San Francisco
ID: 4 | Name: Emily Brown | Dept: HR | Salary: $75,000 | Experience: 3 years | City: Chicago
ID: 5 | Name: James Wilson | Dept: Engineering | Salary: $125,000 | Experience: 6 years | City: New York
ID: 6 | Name: Lisa Anderson | Dept: Marketing | Salary: $110,000 | Experience: 7 years | City: Los Angeles
ID: 7 | Name: Robert Taylor | Dept: Finance | Salary: $135,000 | Experience: 9 years | City: Chicago
ID: 8 | Name: Jennifer Martinez | Dept: HR | Salary: $80,000 | Experience: 4 years | City: San Francisco
ID: 9 | Name: David Thomas | Dept: Finance | Salary: $155,000 | Experience: 11 years | City: New York
ID: 10 | Name: Jessica Garcia | Dep

In [12]:
def ask_database(question):

    # Step 1 - Tell Claude about our database structure
    prompt = f"""
You are a SQL expert. You have access to a SQLite database with this table:

TABLE: employees
COLUMNS:
- id (INTEGER) - unique employee ID
- name (TEXT) - employee full name
- department (TEXT) - department name (Engineering, Marketing, HR, Finance)
- salary (INTEGER) - annual salary in dollars
- years_experience (INTEGER) - years of work experience
- city (TEXT) - city where employee works (New York, Los Angeles, San Francisco, Chicago)

USER QUESTION: {question}

Write a single SQLite SQL query to answer this question.
Return ONLY the SQL query, nothing else. No explanation. No markdown. Just the raw SQL.
"""

    # Step 2 - Ask Claude to write the SQL
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}]
    )

    sql_query = message.content[0].text.strip()

    # Step 3 - Run the SQL on our real database
    cursor.execute(sql_query)
    results = cursor.fetchall()

    # Step 4 - Ask Claude to explain the results in plain English
    explanation_prompt = f"""
The user asked: {question}
The SQL query used was: {sql_query}
The database returned: {results}

Explain the answer in plain English in 2-3 sentences. Be clear and direct.
"""

    explanation = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        messages=[{"role": "user", "content": explanation_prompt}]
    )

    print(f"Question: {question}")
    print(f"SQL Generated: {sql_query}")
    print(f"Raw Results: {results}")
    print(f"Answer: {explanation.content[0].text}")
    print("---")

print("AI Database Assistant ready!")

AI Database Assistant ready!


In [13]:
# Ask 5 different questions to test our AI database assistant
ask_database("Which department has the highest average salary?")
ask_database("How many employees work in New York?")
ask_database("Who is the highest paid employee?")
ask_database("List all engineers ordered by salary from highest to lowest")
ask_database("What is the average salary of employees with more than 7 years of experience?")

Question: Which department has the highest average salary?
SQL Generated: SELECT department, AVG(salary) as average_salary FROM employees GROUP BY department ORDER BY average_salary DESC LIMIT 1;
Raw Results: [('Finance', 145000.0)]
Answer: The Finance department has the highest average salary at $145,000. This means that when you average the salaries of all employees in the Finance department, they earn more on average than employees in any other department.
---
Question: How many employees work in New York?
SQL Generated: SELECT COUNT(*) FROM employees WHERE city = 'New York';
Raw Results: [(3,)]
Answer: # Answer

There are 3 employees who work in New York. This result comes from counting all employee records in the database where the city field is set to 'New York'.
---
Question: Who is the highest paid employee?
SQL Generated: SELECT * FROM employees ORDER BY salary DESC LIMIT 1;
Raw Results: [(3, 'Mike Davis', 'Engineering', 165000, 12, 'San Francisco')]
Answer: # Answer

The high